In [3]:
import os, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import pickle

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(it, desc="", **kw):
        it = list(it); print(f"  {desc}: {len(it)} items"); return it

warnings.filterwarnings("ignore")
np.random.seed(42)

# ══════════════════════════════════════════════════════════════
# 0.  OUTPUT DIRECTORY
# ══════════════════════════════════════════════════════════════
OUT_DIR = "./outputs_baseline"
os.makedirs(OUT_DIR, exist_ok=True)

def out(fname):
    return os.path.join(OUT_DIR, fname)

# ══════════════════════════════════════════════════════════════
# 1.  PAPER CONSTANTS
# ══════════════════════════════════════════════════════════════
ALPHA           = 0.1
GAMMA           = 0.6
EPSILON         = 0.1
LAMBDA_SHAPING  = -100
N_ITERATIONS    = 6      # 6 clips per playlist
N_SUBJECTS      = 32
N_CLIPS         = 40
TRAIN_EPOCHS    = 1000
TARGET_EMOTION  = "happy"
CONVERGE_THRESH = 10     # degrees — below this = converged

# ══════════════════════════════════════════════════════════════
# 2.  GEW — 8 primary emotions + NEUTRAL start state
# ══════════════════════════════════════════════════════════════
GEW_EMOTIONS = {
    "sadness":  [-0.81, -0.71],
    "pride":    [ 0.73,  0.04],
    "interest": [-0.22,  0.71],
    "happy":    [ 0.73,  0.75],
    "guilt":    [-0.21, -0.81],
    "disgust":  [-0.37, -0.06],
    "anger":    [-0.85,  0.69],
    "relief":   [ 0.59, -0.67],
}
NEUTRAL_STATE  = "neutral"
EMOTION_NAMES  = list(GEW_EMOTIONS.keys())       # 8 emotions
ALL_STATES     = [NEUTRAL_STATE] + EMOTION_NAMES  # 9 states
STATE_INDEX    = {s: i for i, s in enumerate(ALL_STATES)}
N_STATES       = len(ALL_STATES)                  # 9

# ── unit helpers ───────────────────────────────────────────────
def _unit(v):
    """Return unit vector, or zero-vector if norm is negligible."""
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    return v / n if n > 1e-9 else v          # returns zero-vec for neutral [0,0]

def _is_neutral_vec(v):
    return np.linalg.norm(np.asarray(v, float)) < 1e-9

GEW_UNIT   = {k: _unit(v) for k, v in GEW_EMOTIONS.items()}
TARGET_VEC = GEW_UNIT[TARGET_EMOTION]

EMOTION_COLORS = {
    "neutral":  "#AAAAAA",
    "happy":    "#FFD700",
    "pride":    "#90EE90",
    "interest": "#00CED1",
    "anger":    "#FF4500",
    "disgust":  "#FF8C00",
    "guilt":    "#9370DB",
    "sadness":  "#4169E1",
    "relief":   "#FF69B4",
}

# ══════════════════════════════════════════════════════════════
# 3.  DATA LOADING
#
#   Pipeline (per-subject):
#     1. z-score each dimension (valence, arousal) with ddof=1
#        across all 40 clips → zero mean, unit variance
#     2. tanh squash → values in (-1, 1) robustly
#     3. L2-normalise each clip vector to the unit circle
#        so it is geometrically compatible with GEW_UNIT centroids
#
#   Step 3 is the key fix: without it the clip vectors have
#   varying magnitudes and _safe_angle_deg / transition() mixes
#   direction and magnitude in an undefined way.
# ══════════════════════════════════════════════════════════════
def _normalise_clips(raw):
    """
    raw : (n_clips, 2) float array  [valence, arousal], arbitrary scale
    """
    # 1. Standard (z-score) scaling per dimension
    mean = raw.mean(axis=0)
    std  = raw.std(axis=0, ddof=1)
    std  = np.where(std < 1e-6, 1.0, std)          
    z    = (raw - mean) / std                      

    # 2. tanh squash -> bounded in (-1, 1) per dimension
    squeezed = np.tanh(z)                          

    # 3. DO NOT L2-normalise. Return the squashed vectors directly.
    return squeezed.astype(float)


def load_deap(csv_path="./deap-dataset/Metadata/participant_ratings.csv"):
    if not os.path.exists(csv_path):
        print(f"  [Error] CSV not found: {csv_path}")
        return None

    df = pd.read_csv(csv_path)
    df = df.sort_values(by=["Participant_id", "Experiment_id"]).reset_index(drop=True)

    subjects = {}
    for sid, group in df.groupby("Participant_id"):
        raw = group[["Valence", "Arousal"]].values.astype(float)   # (40, 2)
        subjects[int(sid)] = _normalise_clips(raw)                 # (40, 2) unit vecs

    print(f"  Loaded DEAP (CSV): {len(subjects)} subjects, "
          f"{next(iter(subjects.values())).shape[0]} clips each.")
    return subjects


def make_synthetic(n_subjects=32, n_clips=40, seed=0):
    """Fallback synthetic data: unit vectors drawn from GEW cluster centres."""
    rng = np.random.default_rng(seed)
    action_vecs = np.array([GEW_UNIT[e] for e in EMOTION_NAMES])
    subjects = {}
    for sid in range(1, n_subjects + 1):
        w      = rng.dirichlet(np.ones(len(action_vecs)) * 0.5)
        chosen = rng.choice(len(action_vecs), size=n_clips, p=w)
        noise  = rng.normal(0, 0.25, (n_clips, 2))
        raw    = action_vecs[chosen] + noise
        # apply same pipeline for consistency
        subjects[sid] = _normalise_clips(raw)
    print(f"  Synthetic DEAP: {n_subjects} subjects, {n_clips} clips each.")
    return subjects

# ══════════════════════════════════════════════════════════════
# 4.  STATE QUANTISATION
# ══════════════════════════════════════════════════════════════
def vec_to_state(vec):
    """
    Map a 2-D valence-arousal vector to the nearest GEW emotion name.

    IMPORTANT: expects a unit vector (or [0,0] for neutral).
    If the vector has negligible norm → NEUTRAL_STATE.
    """
    vec = np.asarray(vec, dtype=float)
    if _is_neutral_vec(vec):
        return NEUTRAL_STATE
    uvec = _unit(vec)                         # safe: already checked norm > 0
    best, best_cos = NEUTRAL_STATE, -np.inf
    for name, gvec in GEW_UNIT.items():
        c = float(np.dot(uvec, gvec))
        if c > best_cos:
            best_cos, best = c, name
    return best

# ══════════════════════════════════════════════════════════════
# 5.  TRANSITION  τ(s, a)  —  paper §III-B
#
#   next_state_vec = normalise( current_state_vec + clip_vec )
#
#   Both inputs must be unit vectors (or zero for neutral).
#   We normalise the *sum* so the result stays on S¹ and
#   angular comparisons remain well-defined across all steps.
#
#   FIX vs original: removed clip() to [-1.5,1.5] before
#   normalising — clipping distorts direction and is redundant
#   once we normalise to the unit circle.
# ══════════════════════════════════════════════════════════════
def transition(vs, va):
    """
    vs : current state vector (or [0,0] for neutral)
    va : clip vector (preserves its magnitude)
    """
    vs = np.asarray(vs, float)
    va = np.asarray(va, float)

    if _is_neutral_vec(vs):
        # Starting from neutral, the state becomes the clip's exact vector
        return va.copy()

    # Standard vector addition as defined in the paper
    summed = vs + va 
    
    # Clip the coordinates to [-1, 1] to keep the state inside the GEW plane
    summed = np.clip(summed, -1.0, 1.0)
    
    return summed
# ══════════════════════════════════════════════════════════════
# 6.  ANGULAR ERROR  (evaluation metric)
#
#   FIX: angular_error([0,0]) was silently returning 0°,
#   masking the true initial distance from target.
#   Now explicitly guard the neutral vector.
# ══════════════════════════════════════════════════════════════
def _safe_angle_deg(v1, v2):
    u1  = _unit(np.asarray(v1, float))
    u2  = _unit(np.asarray(v2, float))
    if _is_neutral_vec(u1) or _is_neutral_vec(u2):
        return 180.0      # undefined direction → worst-case angle
    dot = np.clip(np.dot(u1, u2), -1.0, 1.0)
    return np.degrees(np.arccos(dot))

def angular_error(vs):
    """Angular distance (°) between state vector and TARGET_VEC."""
    return _safe_angle_deg(np.asarray(vs, float), TARGET_VEC)

# ══════════════════════════════════════════════════════════════
# 7.  REWARD  r(s, a)  —  paper §III-C
# ══════════════════════════════════════════════════════════════
def reward_fn(vs, va, s_name):
    vs = np.asarray(vs, float)
    va = np.asarray(va, float)

    # 1. Calculate angles exactly as interpreted from III.C
    theta1 = _safe_angle_deg(TARGET_VEC, va)
    theta2 = _safe_angle_deg(vs, va)

    # 2. Strict boolean reward: +1 if combined angle <= 180, else 0
    return 1.0 if (theta1 + theta2) <= 180.0 else 0.0


def shaping_fn(s_name, sp_name):
    # Penalize stagnation, UNLESS it's the target emotion
    if s_name == sp_name and s_name != TARGET_EMOTION:
        return LAMBDA_SHAPING
    return 0.0

# ══════════════════════════════════════════════════════════════
# 8.  Q-LEARNING AGENT
# ══════════════════════════════════════════════════════════════
class QAgent:
    def __init__(self, n_states, n_clips):
        self.Q = np.random.uniform(-0.01, 0.01, (n_states, n_clips))

    def act(self, state_idx, n_clips, explore=True):
        if explore and np.random.rand() < EPSILON:
            return np.random.randint(n_clips)
        return int(np.argmax(self.Q[state_idx, :n_clips]))

    def update(self, state_idx, clip_idx, r, phi, next_state_idx, n_clips):
        best_next = float(np.max(self.Q[next_state_idx, :n_clips]))
        target    = r + GAMMA * best_next + phi
        self.Q[state_idx, clip_idx] = (
            (1.0 - ALPHA) * self.Q[state_idx, clip_idx] + ALPHA * target
        )

# ══════════════════════════════════════════════════════════════
# 9.  LEAVE-ONE-SUBJECT-OUT CROSS-VALIDATION
#
#   FIX: evaluation now starts from NEUTRAL [0,0] consistently,
#   matching both the paper and the training distribution.
#   The original code had a hardcoded anger vector [-1.50, 0.84]
#   for evaluation while training from neutral — a train/test
#   mismatch that inflated angular errors artificially.
#
#   If you want to evaluate from anger specifically, set
#   EVAL_START_EMOTION = "anger"  (see constant below).
#   Training will then also use a matched angry starting point.
# ══════════════════════════════════════════════════════════════
EVAL_FROM_NEUTRAL = False        # True  → paper-faithful neutral start
                                   # False → start from EVAL_START_EMOTION
EVAL_START_EMOTION = "anger"       # used only when EVAL_FROM_NEUTRAL = False


def _make_start_vec(rng=None):
    """Return the starting valence-arousal vector for an episode."""
    if EVAL_FROM_NEUTRAL:
        return np.array([0.0, 0.0])
    else:
        # Unit vector of the chosen emotion + tiny noise so it isn't
        # exactly at the centroid (more realistic)
        base = GEW_UNIT[EVAL_START_EMOTION].copy()
        if rng is not None:
            base = _unit(base + rng.normal(0, 0.05, 2))
        return base


def run_loso(subjects, use_shaping=True, verbose=True):
    sids = sorted(subjects.keys())
    errors, state_paths, vec_paths = {}, {}, {}
    rng = np.random.default_rng(0)

    iterator = tqdm(sids, desc="LOSO", disable=not verbose)
    for test_sid in iterator:
        train_sids = [s for s in sids if s != test_sid]

        agent = QAgent(N_STATES, N_CLIPS)

        # ── TRAINING ─────────────────────────────────────────
        for epoch in range(TRAIN_EPOCHS):
            for tr_sid in train_sids:
                clips   = subjects[tr_sid]   # (40, 2) unit vectors
                n_clips = len(clips)

                vs    = _make_start_vec()          # consistent start
                s     = vec_to_state(vs)
                s_idx = STATE_INDEX[s]

                for step in range(N_ITERATIONS):
                    # 1. ε-greedy clip selection
                    clip_idx = agent.act(s_idx, n_clips, explore=True)

                    # 2. Clip's unit vector
                    va = clips[clip_idx]              # already unit vec

                    # 3. Transition: unit vec → unit vec
                    vsp   = transition(vs, va)
                    sp    = vec_to_state(vsp)
                    sp_idx = STATE_INDEX[sp]

                    # 4. Reward + optional shaping
                    r   = reward_fn(vs, va, s)
                    phi = shaping_fn(s, sp) if use_shaping else 0.0

                    # 5. Q-update
                    agent.update(s_idx, clip_idx, r, phi, sp_idx, n_clips)

                    # 6. Advance state
                    vs, s, s_idx = vsp, sp, sp_idx

        # ── GREEDY EVALUATION ────────────────────────────────
        clips   = subjects[test_sid]
        n_clips = len(clips)

        vs    = _make_start_vec()                     # no randomness at eval
        s     = vec_to_state(vs)
        s_idx = STATE_INDEX[s]

        # Initial angular error before any clip is played
        # FIX: angular_error([0,0]) now returns 180° (worst case)
        #      instead of the previous silent 0°.
        errs   = []
        states = [s]
        vecs   = [vs.copy()]

        for _ in range(N_ITERATIONS):
            clip_idx = agent.act(s_idx, n_clips, explore=False)
            va       = clips[clip_idx]
            vs       = transition(vs, va)             # unit vec out
            s        = vec_to_state(vs)
            s_idx    = STATE_INDEX[s]
            errs.append(angular_error(vs))
            states.append(s)
            vecs.append(vs.copy())

        errors[test_sid]      = np.array(errs)
        state_paths[test_sid] = states
        vec_paths[test_sid]   = vecs

    return errors, state_paths, vec_paths

# ══════════════════════════════════════════════════════════════
# 10.  TABLE I
# ══════════════════════════════════════════════════════════════
def compute_table1(errors):
    sids  = sorted(errors.keys())
    arr   = np.array([errors[s] for s in sids])
    final = arr[:, -1]
    converged = final < CONVERGE_THRESH

    S = arr[ converged]
    F = arr[~converged]
    T = arr

    col_groups = [slice(0, 2), slice(2, 3), slice(3, 4), slice(4, 5), slice(5, 6)]
    headers    = ["Clip 1-2", "3", "4", "5", "6"]

    rows = {}
    for lbl, sub in [("T", T), ("S", S), ("F", F)]:
        row = []
        for sl in col_groups:
            v = sub[:, sl].flatten()
            row.append(f"{v.mean():.1f}({v.std():.1f})" if len(v) else "N/A")
        rows[lbl] = row

    df = pd.DataFrame(rows, index=headers).T
    df.index.name = "Group"
    return df, np.where(converged)[0], np.where(~converged)[0]


def print_table1(errors):
    df, succ, fail = compute_table1(errors)
    sids  = sorted(errors.keys())
    arr   = np.array([errors[s] for s in sids])
    final = arr[:, -1]

    S_ids = [sids[i] for i in succ]
    F_ids = [sids[i] for i in fail]

    print("\n" + "═" * 65)
    print("  TABLE I  —  Mean Angular Error (std)  |  T=Total S=Success F=Fail")
    print("  Paper target: 19 S-subjects, clip-6 ≈11.3°, overall ≈57°")
    print("═" * 65)
    print(df.to_string())
    if len(succ):
        print(f"\n  Converged (<{CONVERGE_THRESH}°): {len(succ)}/32  "
              f"(clip-6 mean {final[succ].mean():.1f}°)")
    else:
        print(f"\n  Converged: 0/32")
    if len(fail):
        print(f"  Failed: {len(fail)}/32  "
              f"(clip-6 mean {final[fail].mean():.1f}°)")
    else:
        print(f"  Failed: 0/32")
    print(f"  Overall clip-6: {final.mean():.1f}° ± {final.std():.1f}°")
    return df, S_ids, F_ids

# ══════════════════════════════════════════════════════════════
# 11.  FIGURE 2  —  Angular Error vs Iterations (with/without shaping)
# ══════════════════════════════════════════════════════════════
def plot_fig2(err_with, err_without):
    iters   = np.arange(1, N_ITERATIONS + 1)
    mean_w  = np.mean([err_with[s]    for s in sorted(err_with)],   axis=0)
    mean_wo = np.mean([err_without[s] for s in sorted(err_without)], axis=0)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(iters, mean_wo, color="steelblue",  lw=2, ls="--",
            label="Without reward shaping")
    ax.plot(iters, mean_w,  color="darkorange", lw=2, ls="-.",
            label="With reward shaping")

    peak = int(np.argmax(mean_wo))
    ax.annotate("The angular error gets stuck\nto a local minimum",
                xy=(peak + 1, mean_wo[peak]),
                xytext=(max(peak - 0.8, 0.3), mean_wo[peak] + 22),
                fontsize=8, color="steelblue",
                arrowprops=dict(arrowstyle="->", color="steelblue"))
    ax.annotate("The angular error\nsteadily decreases",
                xy=(5, mean_w[4]),
                xytext=(3.2, mean_w[4] - 25),
                fontsize=8, color="darkorange",
                arrowprops=dict(arrowstyle="->", color="darkorange"))

    ax.set_xlabel("Number of iterations")
    ax.set_ylabel("Angular Error in Emotion Transition (degrees)")
    ax.set_title("Angular Error vs Iterations  (paper Fig. 2)")
    ax.set_xlim(0, N_ITERATIONS + 0.5)
    ax.set_ylim(0, 230)
    ax.legend()
    plt.tight_layout()
    plt.savefig(out("fig2_angular_error.png"), dpi=150)
    plt.close()
    print("  Saved: fig2_angular_error.png")

# ══════════════════════════════════════════════════════════════
# 12.  FIGURE 3  —  Error Bar Plot
# ══════════════════════════════════════════════════════════════
def plot_fig3(err_with):
    arr  = np.array([err_with[s] for s in sorted(err_with)])
    mean = arr.mean(axis=0)
    se   = arr.std(axis=0) / np.sqrt(len(arr))
    iters = np.arange(1, N_ITERATIONS + 1)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(iters, mean, yerr=se, fmt="-o", color="black",
                capsize=5, lw=1.8, ms=5, elinewidth=1.5)
    ax.set_xlabel("# of iterations")
    ax.set_ylabel("Angular Error (degrees)")
    ax.set_title("Error Bar Plot  (paper Fig. 3)")
    ax.set_xlim(0.5, 6.5)
    ax.set_xticks(iters)
    plt.tight_layout()
    plt.savefig(out("fig3_error_bar.png"), dpi=150)
    plt.close()
    print("  Saved: fig3_error_bar.png")

# ══════════════════════════════════════════════════════════════
# 13.  FIGURE 4  —  Two-subject emotion paths
#
#   FIX: initial error is now angular_error(vs_initial) which
#   returns 180° for neutral start (proper worst-case baseline)
#   instead of the previous always-0° from angular_error([0,0]).
# ══════════════════════════════════════════════════════════════
def plot_fig4(err_with, state_paths, S_ids, F_ids):
    if not S_ids or not F_ids:
        print("  Skipping fig4 (need ≥1 converged AND ≥1 failed subject)")
        return

    sids  = sorted(err_with.keys())
    arr   = np.array([err_with[s] for s in sids])
    final = arr[:, -1]

    S_final  = [(s, final[sids.index(s)]) for s in S_ids]
    F_final  = [(s, final[sids.index(s)]) for s in F_ids]
    good_sid = min(S_final, key=lambda x: x[1])[0]
    bad_sid  = max(F_final, key=lambda x: x[1])[0]

    fig, ax = plt.subplots(figsize=(9, 5))
    iters = np.arange(N_ITERATIONS + 1)

    vs_init = _make_start_vec()
    init_err = angular_error(vs_init)           # FIX: true initial error

    for sid, color, label in [(good_sid, "tab:blue",   f"Subject {good_sid} (converged)"),
                               (bad_sid,  "tab:orange", f"Subject {bad_sid}  (oscillating)")]:
        path      = state_paths[sid]
        errs_full = [init_err] + list(err_with[sid])    # FIX: use real init_err
        ax.plot(iters, errs_full, "o--", color=color, lw=1.8, label=label)
        for i, (err, em) in enumerate(zip(errs_full, path)):
            c = EMOTION_COLORS.get(em, "gray")
            ax.scatter(i, err, color=c, s=70, zorder=5)
            ax.annotate(em.capitalize(), (i, err),
                        textcoords="offset points", xytext=(5, 4),
                        fontsize=7, color=c, fontweight="bold")

    ax.set_xlabel("# of Iterations")
    ax.set_ylabel("Angular Error (degrees)")
    ax.set_title("Angular Error and Emotional State — Two Subjects  (paper Fig. 4)")
    ax.set_ylim(-5, 200)
    ax.set_xticks(iters)
    ax.legend()
    plt.tight_layout()
    plt.savefig(out("fig4_two_subjects.png"), dpi=150)
    plt.close()
    print("  Saved: fig4_two_subjects.png")

# ══════════════════════════════════════════════════════════════
# 14.  GEW WHEEL HELPER
# ══════════════════════════════════════════════════════════════
def _draw_gew_wheel(ax, alpha_bg=0.12):
    ax.fill_betweenx([0, 1.2],  0,    1.2,  color="#FFFACD", alpha=alpha_bg)
    ax.fill_betweenx([0, 1.2], -1.2,  0,    color="#FFE4E1", alpha=alpha_bg)
    ax.fill_betweenx([-1.2, 0], -1.2, 0,    color="#E0E0FF", alpha=alpha_bg)
    ax.fill_betweenx([-1.2, 0],  0,   1.2,  color="#E8FFE8", alpha=alpha_bg)

    circle = plt.Circle((0, 0), 1.0, color="silver", fill=False, lw=1.0, ls="--")
    ax.add_patch(circle)
    ax.axhline(0, color="gray", lw=0.6, ls=":")
    ax.axvline(0, color="gray", lw=0.6, ls=":")

    for name, uvec in GEW_UNIT.items():
        x, y = uvec * 1.08
        c = EMOTION_COLORS[name]
        ax.scatter(*uvec, color=c, s=55, zorder=6, edgecolors="k", lw=0.4)
        ax.text(x, y, name[:4].capitalize(), fontsize=6, ha="center",
                va="center", color=c, fontweight="bold")

    ax.scatter(*TARGET_VEC, marker="*", s=180, color="gold",
               edgecolors="darkorange", lw=0.8, zorder=8)
    ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35)
    ax.set_aspect("equal"); ax.tick_params(labelsize=6)
    ax.set_xlabel("Valence →", fontsize=7)
    ax.set_ylabel("Arousal →", fontsize=7)
    for tx, ty, txt in [(0.65, 1.22, "Joy"), (-0.65, 1.22, "Anger"),
                         (-0.65,-1.22, "Depressive"), (0.65,-1.22, "Hopeful")]:
        ax.text(tx, ty, txt, fontsize=6, ha="center", color="dimgray", style="italic")

# ══════════════════════════════════════════════════════════════
# 15.  INDIVIDUAL SUBJECT TRAJECTORY PLOT
# ══════════════════════════════════════════════════════════════
def plot_subject_trajectory(sid, vec_paths, err_with, converged, save=True):
    vecs  = vec_paths[sid]
    vs_init = _make_start_vec()
    init_err = angular_error(vs_init)            # FIX: real initial error
    errs  = [init_err] + list(err_with[sid])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    status = "CONVERGED" if converged else "FAILED"
    fig.suptitle(f"Subject {sid:02d} — {status}", fontsize=11, fontweight="bold",
                 color="green" if converged else "red")

    ax = axes[0]
    _draw_gew_wheel(ax)
    cmap = plt.get_cmap("plasma")
    for i in range(len(vecs) - 1):
        v0, v1 = np.array(vecs[i]), np.array(vecs[i + 1])
        # FIX: skip drawing arrow FROM neutral (zero vector) — just show dot
        frac   = i / max(len(vecs) - 2, 1)
        if not _is_neutral_vec(v0):
            ax.annotate("", xy=v1, xytext=v0,
                        arrowprops=dict(arrowstyle="-|>", color=cmap(frac),
                                        lw=1.5, mutation_scale=12))
        ax.scatter(*v0 if not _is_neutral_vec(v0) else [0,0],
                   color=cmap(frac), s=40, zorder=7, edgecolors="k", lw=0.3)

    ax.scatter(0, 0, s=80, color="lime", edgecolors="k", lw=0.6, zorder=9,
               marker="o", label="Start (neutral)")
    if not _is_neutral_vec(vecs[-1]):
        ax.scatter(*vecs[-1], s=100, color="red", edgecolors="k", lw=0.6,
                   zorder=9, marker="X", label="End")
    for i, v in enumerate(vecs):
        px = v[0] if not _is_neutral_vec(v) else 0.0
        py = v[1] if not _is_neutral_vec(v) else 0.0
        ax.text(px + 0.05, py + 0.05, str(i), fontsize=7, color="navy")
    ax.set_title("GEW Emotion Trajectory", fontsize=9)
    ax.legend(fontsize=7, loc="lower right")

    ax2 = axes[1]
    iters = np.arange(len(errs))
    col   = "green" if converged else "tomato"
    ax2.plot(iters, errs, "o-", color=col, lw=2, ms=6)
    ax2.axhline(CONVERGE_THRESH, color="gray", ls="--", lw=1,
                label=f"Convergence threshold ({CONVERGE_THRESH}°)")
    ax2.fill_between(iters, 0, errs, alpha=0.12, color=col)
    ax2.set_xlabel("Iteration (0 = initial)", fontsize=9)
    ax2.set_ylabel("Angular Error (°)", fontsize=9)
    ax2.set_title("Angular Error vs Iteration", fontsize=9)
    ax2.set_xticks(iters)
    ax2.set_ylim(0, 200)
    ax2.legend(fontsize=7)

    states = [vec_to_state(v) for v in vecs]
    for i, (err, st) in enumerate(zip(errs, states)):
        c = EMOTION_COLORS.get(st, "gray")
        ax2.annotate(st[:4].capitalize(), (i, err),
                     textcoords="offset points", xytext=(4, 5),
                     fontsize=7, color=c, fontweight="bold")

    plt.tight_layout()
    if save:
        plt.savefig(out(f"trajectory_subject_{sid:02d}.png"), dpi=130)
        plt.close()
    return fig

# ══════════════════════════════════════════════════════════════
# 16.  ALL-SUBJECTS GRID
# ══════════════════════════════════════════════════════════════
def plot_all_trajectories_grid(err_with, vec_paths, S_ids, F_ids):
    sids  = sorted(err_with.keys())
    ncols = 8
    nrows = int(np.ceil(len(sids) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 2.8, nrows * 2.8))
    fig.suptitle("GEW Emotion Trajectories — All Subjects\n"
                 "★ = target (Happy)  |  green border = converged  |  red = failed",
                 fontsize=11, y=1.01)

    cmap = plt.get_cmap("plasma")
    for idx, sid in enumerate(sids):
        row, col = divmod(idx, ncols)
        ax = axes[row][col] if nrows > 1 else axes[col]
        _draw_gew_wheel(ax, alpha_bg=0.08)
        vecs = vec_paths[sid]
        conv = sid in S_ids
        for i in range(len(vecs) - 1):
            v0, v1 = np.array(vecs[i]), np.array(vecs[i + 1])
            frac   = i / max(len(vecs) - 2, 1)
            if not _is_neutral_vec(v0):                 # FIX: skip zero-vec arrow
                ax.annotate("", xy=v1, xytext=v0,
                            arrowprops=dict(arrowstyle="-|>", color=cmap(frac),
                                            lw=1.2, mutation_scale=10))
        ax.scatter(0, 0, s=30, color="lime", zorder=9, edgecolors="k", lw=0.3)
        if not _is_neutral_vec(vecs[-1]):
            ax.scatter(*vecs[-1], s=40, color="red", zorder=9, marker="X",
                       edgecolors="k", lw=0.3)
        border_col = "#00AA00" if conv else "#CC0000"
        for spine in ax.spines.values():
            spine.set_edgecolor(border_col); spine.set_linewidth(2.0)
        ax.set_title(f"S{sid:02d}", fontsize=7, pad=1,
                     color="green" if conv else "red", fontweight="bold")
        ax.set_xticklabels([]); ax.set_yticklabels([])
        ax.set_xlabel(""); ax.set_ylabel("")

    for idx in range(len(sids), nrows * ncols):
        row, col = divmod(idx, ncols)
        ax = axes[row][col] if nrows > 1 else axes[col]
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(out("fig_all_trajectories_grid.png"), dpi=130, bbox_inches="tight")
    plt.close()
    print("  Saved: fig_all_trajectories_grid.png")

# ══════════════════════════════════════════════════════════════
# 17.  REGRET PLOT
# ══════════════════════════════════════════════════════════════
def plot_regret(err_with, err_without):
    sids   = sorted(err_with.keys())
    reg_w  = np.cumsum(np.mean([err_with[s]    for s in sids], axis=0))
    reg_wo = np.cumsum(np.mean([err_without[s] for s in sids], axis=0))

    iters = np.arange(1, N_ITERATIONS + 1)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(iters, reg_wo, "b--", lw=2, label="Without reward shaping")
    ax.plot(iters, reg_w,  "r-.", lw=2, label="With reward shaping")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Cumulative Angular Error (Regret)")
    ax.set_title("Regret vs Iterations  (paper §IV-A)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out("fig_regret.png"), dpi=150)
    plt.close()
    print("  Saved: fig_regret.png")

# ══════════════════════════════════════════════════════════════
# 18.  TEXT REPORT
# ══════════════════════════════════════════════════════════════
def save_report(err_with, state_paths, S_ids, F_ids, table_df):
    sids  = sorted(err_with.keys())
    arr   = np.array([err_with[s] for s in sids])
    final = arr[:, -1]

    lines = ["=" * 72,
             "  RL Music Emotion Management — DEAP Experiment Report",
             "  Reference: Dutta et al., IEEE EMBC 2020",
             "=" * 72,
             f"\n  Subjects evaluated      : {len(sids)}",
             f"  Converged (<{CONVERGE_THRESH}°)  : {len(S_ids)}  ({100*len(S_ids)/len(sids):.1f}%)",
             f"  Did NOT converge        : {len(F_ids)}  ({100*len(F_ids)/len(sids):.1f}%)",
             f"\n  Overall clip-6 error    : {final.mean():.1f}° ± {final.std():.1f}°",
             f"  Paper target            : ≈57° overall, ≈11.3° S-group, ≈123.7° F-group"]

    if S_ids:
        S_f = final[[sids.index(s) for s in S_ids]]
        lines.append(f"  S-group clip-6 error    : {S_f.mean():.1f}° ± {S_f.std():.1f}°")
    if F_ids:
        F_f = final[[sids.index(s) for s in F_ids]]
        lines.append(f"  F-group clip-6 error    : {F_f.mean():.1f}° ± {F_f.std():.1f}°")

    lines += ["\n\n--- TABLE I ---", table_df.to_string(),
              "\n\n--- PER-SUBJECT DETAIL ---",
              f"{'SID':>4}  {'Status':>10}  {'FinalErr':>9}  Emotion Path",
              "-" * 72]

    for i, sid in enumerate(sids):
        status = "CONVERGED" if sid in S_ids else "FAILED"
        path   = " → ".join(state_paths[sid])
        lines.append(f"{sid:>4}  {status:>10}  {final[i]:>8.1f}°  {path}")

    report = "\n".join(lines)
    with open(out("experiment_report.txt"), "w", encoding="utf-8") as f:
        f.write(report)
    print("  Saved: experiment_report.txt")
    print("\n" + report)

# ══════════════════════════════════════════════════════════════
# 19.  MAIN
# ══════════════════════════════════════════════════════════════
def main():
    print("\n" + "═" * 65)
    print("  RL Music Emotion Management — DEAP (Dutta et al. 2020)")
    print("  Q-table: 9 states × 40 clips  (clips ARE the actions)")
    print("═" * 65)

    print("\n[1] Loading data …")
    subjects = load_deap()
    if subjects is None:
        print("  CSV not found → using synthetic data")
        subjects = make_synthetic()

    print("\n[2] LOSO — WITH reward shaping …")
    err_with, state_paths, vec_paths = run_loso(subjects, use_shaping=True)

    print("\n[3] LOSO — WITHOUT reward shaping (for Fig. 2 / regret) …")
    err_without, _, _ = run_loso(subjects, use_shaping=False)

    print("\n[4] Results …")
    table_df, S_ids, F_ids = print_table1(err_with)
    print(f"\n  Converged subject IDs : {sorted(S_ids)}")
    print(f"  Failed    subject IDs : {sorted(F_ids)}")

    print("\n[5] Paper figures …")
    plot_fig2(err_with, err_without)
    plot_fig3(err_with)
    plot_fig4(err_with, state_paths, S_ids, F_ids)
    plot_regret(err_with, err_without)

    print("\n[6] Per-subject trajectory plots …")
    for sid in tqdm(sorted(err_with.keys()), desc="Trajectories"):
        plot_subject_trajectory(sid, vec_paths, err_with, sid in S_ids)
    print(f"  Saved {len(subjects)} trajectory plots.")

    print("\n[7] All-subjects grid …")
    plot_all_trajectories_grid(err_with, vec_paths, S_ids, F_ids)

    print("\n[8] Writing report …")
    save_report(err_with, state_paths, S_ids, F_ids, table_df)

    print("\n" + "═" * 65)
    print("  ALL OUTPUTS saved to ./outputs8")
    print("─" * 65)
    print("  fig2_angular_error.png")
    print("  fig3_error_bar.png")
    print("  fig4_two_subjects.png")
    print("  fig_regret.png")
    print("  trajectory_subject_NN.png   (32 files)")
    print("  fig_all_trajectories_grid.png")
    print("  experiment_report.txt")
    print("═" * 65)


if __name__ == "__main__":
    main()


═════════════════════════════════════════════════════════════════
  RL Music Emotion Management — DEAP (Dutta et al. 2020)
  Q-table: 9 states × 40 clips  (clips ARE the actions)
═════════════════════════════════════════════════════════════════

[1] Loading data …
  Loaded DEAP (CSV): 32 subjects, 40 clips each.

[2] LOSO — WITH reward shaping …


LOSO: 100%|██████████| 32/32 [11:34<00:00, 21.71s/it]



[3] LOSO — WITHOUT reward shaping (for Fig. 2 / regret) …


LOSO: 100%|██████████| 32/32 [13:15<00:00, 24.85s/it]



[4] Results …

═════════════════════════════════════════════════════════════════
  TABLE I  —  Mean Angular Error (std)  |  T=Total S=Success F=Fail
  Paper target: 19 S-subjects, clip-6 ≈11.3°, overall ≈57°
═════════════════════════════════════════════════════════════════
         Clip 1-2           3           4           5           6
Group                                                            
T      51.5(43.0)  43.1(50.2)  35.2(41.1)  36.5(47.3)  42.3(56.8)
S      38.0(33.7)    8.5(8.9)    3.6(6.2)   6.7(11.8)    0.8(0.0)
F      63.3(46.6)  73.6(51.9)  63.0(38.6)  62.9(51.1)  78.9(56.8)

  Converged (<10°): 15/32  (clip-6 mean 0.8°)
  Failed: 17/32  (clip-6 mean 78.9°)
  Overall clip-6: 42.3° ± 56.8°

  Converged subject IDs : [2, 6, 8, 10, 15, 16, 17, 18, 19, 21, 22, 23, 26, 27, 30]
  Failed    subject IDs : [1, 3, 4, 5, 7, 9, 11, 12, 13, 14, 20, 24, 25, 28, 29, 31, 32]

[5] Paper figures …
  Saved: fig2_angular_error.png
  Saved: fig3_error_bar.png
  Saved: fig4_two_subjec

Trajectories: 100%|██████████| 32/32 [00:20<00:00,  1.58it/s]


  Saved 32 trajectory plots.

[7] All-subjects grid …
  Saved: fig_all_trajectories_grid.png

[8] Writing report …
  Saved: experiment_report.txt

  RL Music Emotion Management — DEAP Experiment Report
  Reference: Dutta et al., IEEE EMBC 2020

  Subjects evaluated      : 32
  Converged (<10°)  : 15  (46.9%)
  Did NOT converge        : 17  (53.1%)

  Overall clip-6 error    : 42.3° ± 56.8°
  Paper target            : ≈57° overall, ≈11.3° S-group, ≈123.7° F-group
  S-group clip-6 error    : 0.8° ± 0.0°
  F-group clip-6 error    : 78.9° ± 56.8°


--- TABLE I ---
         Clip 1-2           3           4           5           6
Group                                                            
T      51.5(43.0)  43.1(50.2)  35.2(41.1)  36.5(47.3)  42.3(56.8)
S      38.0(33.7)    8.5(8.9)    3.6(6.2)   6.7(11.8)    0.8(0.0)
F      63.3(46.6)  73.6(51.9)  63.0(38.6)  62.9(51.1)  78.9(56.8)


--- PER-SUBJECT DETAIL ---
 SID      Status   FinalErr  Emotion Path
--------------------------------